A SEGUIR ESTÃO OS CÓDIGOS PARA A APLICAÇÃO DAS EQUAÇÕES DE PREDIÇÃO

Esta célula lê uma base de dados clínica, separa os indivíduos de acordo com a presença ou ausência de fator de risco e calcula valores de VOP estimados a partir das diferentes equações preditivas. Para cada grupo, são geradas colunas correspondentes às equações aplicáveis, considerando combinações de idade, pressão arterial média e sexo quando pertinente. Em seguida, os valores estimados são comparados com a VOP observada por meio do cálculo do coeficiente de determinação (R²) e do erro quadrático médio da raiz (RMSE), utilizando apenas registros completos. As métricas são exibidas para cada modelo e grupo, permitindo avaliar o desempenho das equações na predição da VOP. O código também inclui tratamento de erros para ausência de colunas obrigatórias ou do arquivo de entrada, e mantém comentado o bloco de salvamento dos resultados em arquivo externo.

In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

ARQUIVO_ENTRADA = 'BANCOS/MONICA_PADRAO.xlsx'
ARQUIVO_SAIDA = 'monica_resultado.xlsx'

metricas_individuais = []
metricas_globais = []

def calcular_metricas(df, coluna_predita, nome_grupo):
    df_clean = df.dropna(subset=['VOP', coluna_predita])

    if len(df_clean) < 2:
        return

    y_true = df_clean['VOP'].values
    y_pred = df_clean[coluna_predita].values
    residuos = y_true - y_pred

    metricas_individuais.append({
        "Grupo": nome_grupo,
        "Modelo": coluna_predita,
        "N": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "STD": np.std(residuos, ddof=1)
    })


def calcular_metricas_globais(lista_df, lista_colunas, nome_conjunto):
    y_true_total = []
    y_pred_total = []

    for df, col in zip(lista_df, lista_colunas):
        df_clean = df.dropna(subset=['VOP', col])
        y_true_total.append(df_clean['VOP'].values)
        y_pred_total.append(df_clean[col].values)

    y_true_total = np.concatenate(y_true_total)
    y_pred_total = np.concatenate(y_pred_total)

    residuos = y_true_total - y_pred_total

    metricas_globais.append({
        "Conjunto": nome_conjunto,
        "N_total": len(y_true_total),
        "R2_total": r2_score(y_true_total, y_pred_total),
        "RMSE_total": np.sqrt(mean_squared_error(y_true_total, y_pred_total)),
        "STD_total": np.std(residuos, ddof=1)
    })

try:
    df = pd.read_excel(ARQUIVO_ENTRADA)

    df0 = df[df['FATOR_RISCO'] == 0].copy()
    df1 = df[df['FATOR_RISCO'] == 1].copy()

    age0, MAP0, sexo0 = df0['IDADE'], df0['PAM'], df0['SEXO']
    age1, MAP1, sexo1 = df1['IDADE'], df1['PAM'], df1['SEXO']

    fator_sexo1_pos = np.where(sexo1 == 1, 0.0929, -0.0929)

    df0["Eq_ufes_sem_sexo_(-)"] = (
        4.43 - 0.052 * age0 + 0.00079 * age0**2
        + 0.00036 * age0 * MAP0 + 0.0235 * MAP0
    )

    df0["Eq_europa_(-)"] = (
        4.62 - 0.13 * age0 + 0.0018 * age0**2
        + 0.0006 * age0 * MAP0 + 0.0284 * MAP0
    )

    df0["Eq_ufes_com_sexo_(-)"] = (
        5.56 - 0.065 * age0 + 0.00071 * age0**2
        + 0.00058 * age0 * MAP0 + 0.0101 * MAP0
        + np.where(sexo0 == 1, 0.188, -0.188)
    )

    df1["Eq_ufes_sem_sexo_(+)"] = (
        6.52 - 0.173 * age1 + 0.002319 * age1**2
        - 0.00001355 * age1**2 * MAP1
        + 0.001430 * age1 * MAP1 + 0.005191 * MAP1
    )

    df1["Eq_ufes_com_sexo_(+)"] = (
        6.73 - 0.171 * age1 + 0.002267 * age1**2
        - 0.00001331 * age1**2 * MAP1
        + 0.001447 * age1 * MAP1 + 0.002360 * MAP1
        + fator_sexo1_pos
    )

    df1["Eq_europa_(+)"] = (
        9.58 - 0.402 * age1 + 0.004560 * age1**2
        - 0.0000262 * age1**2 * MAP1
        + 0.003176 * age1 * MAP1 - 0.01832 * MAP1
    )

    calcular_metricas(df0, "Eq_ufes_sem_sexo_(-)", "Sem Fator (UFES SEM -)")
    calcular_metricas(df0, "Eq_europa_(-)", "Sem Fator (EUROPA -)")
    calcular_metricas(df0, "Eq_ufes_com_sexo_(-)", "Sem Fator (UFES COM -)")

    calcular_metricas(df1, "Eq_ufes_sem_sexo_(+)", "Com Fator (UFES SEM +)")
    calcular_metricas(df1, "Eq_ufes_com_sexo_(+)", "Com Fator (UFES COM +)")
    calcular_metricas(df1, "Eq_europa_(+)", "Com Fator (EUROPA +)")

    calcular_metricas_globais(
        [df0, df1],
        ["Eq_ufes_sem_sexo_(-)", "Eq_ufes_sem_sexo_(+)"],
        "UFES – Sem Sexo (Global)"
    )

    calcular_metricas_globais(
        [df0, df1],
        ["Eq_ufes_com_sexo_(-)", "Eq_ufes_com_sexo_(+)"],
        "UFES – Com Sexo (Global)"
    )

    calcular_metricas_globais(
        [df0, df1],
        ["Eq_europa_(-)", "Eq_europa_(+)"],
        "EUROPA (Global)"
    )

    df_ind = pd.DataFrame(metricas_individuais).round(4)
    df_glob = pd.DataFrame(metricas_globais).round(4)

    print("\n=== MÉTRICAS INDIVIDUAIS ===")
    print(df_ind)

    print("\n=== MÉTRICAS GLOBAIS ===")
    print(df_glob)

    # Exportação opcional
    # with pd.ExcelWriter(ARQUIVO_SAIDA, engine='xlsxwriter') as writer:
    #     df.to_excel(writer, sheet_name='Dados_Originais', index=False)
    #     df0.to_excel(writer, sheet_name='Sem_Fator', index=False)
    #     df1.to_excel(writer, sheet_name='Com_Fator', index=False)
    #     df_ind.to_excel(writer, sheet_name='Metricas_Individuais', index=False)
    #     df_glob.to_excel(writer, sheet_name='Metricas_Globais', index=False)

except Exception as e:
    print(f"Erro: {e}")



=== MÉTRICAS INDIVIDUAIS ===
                    Grupo                Modelo     N      R2    RMSE     STD
0  Sem Fator (UFES SEM -)  Eq_ufes_sem_sexo_(-)   203 -1.3422  2.3442  1.4061
1    Sem Fator (EUROPA -)         Eq_europa_(-)   203 -1.3786  2.3624  1.4077
2  Sem Fator (UFES COM -)  Eq_ufes_com_sexo_(-)   203 -1.2655  2.3055  1.3938
3  Com Fator (UFES SEM +)  Eq_ufes_sem_sexo_(+)  1299 -0.6465  2.8286  1.8120
4  Com Fator (UFES COM +)  Eq_ufes_com_sexo_(+)  1299 -0.6152  2.8015  1.8060
5    Com Fator (EUROPA +)         Eq_europa_(+)  1299 -0.3081  2.5212  1.8100

=== MÉTRICAS GLOBAIS ===
                   Conjunto  N_total  R2_total  RMSE_total  STD_total
0  UFES – Sem Sexo (Global)     1502   -0.6283      2.7681     1.7651
1  UFES – Com Sexo (Global)     1502   -0.5951      2.7397     1.7586
2           EUROPA (Global)     1502   -0.3285      2.5003     1.7613


Este trecho gera um gráfico que apresenta a relação entre os valores reais e os valores preditos de VOP por meio de um diagrama de dispersão. Cada ponto representa uma observação, permitindo visualizar o grau de concordância entre o valor medido e o estimado. A linha de identidade (Y = X) indica o cenário de predição perfeita, enquanto a linha de tendência linear mostra o padrão médio da relação entre as variáveis. A igualdade das escalas nos eixos facilita a avaliação visual de desvios sistemáticos, superestimação ou subestimação dos valores preditos em relação aos valores reais.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr

# --- CONFIGURAÇÃO ---
NOME_ARQUIVO_EXCEL = 'dados_teste_R2.xlsx'
NOME_ABA = 0

COLUNA_REAL = 'VOP_MEDIDO'
COLUNA_PRED = 'PWVEst_PAM_Clinica'

# --- EXECUÇÃO ---
try:
    df = pd.read_excel(NOME_ARQUIVO_EXCEL, sheet_name=NOME_ABA)
    df_clean = df.dropna(subset=[COLUNA_REAL, COLUNA_PRED]).copy()

    y_true = df_clean[COLUNA_REAL]
    y_pred = df_clean[COLUNA_PRED]

    if len(y_true) < 2:
        print("Erro: Dados insuficientes.")
    else:
        # MÉTRICAS
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        # Correlação de Pearson
        pearson_r, pearson_p = pearsonr(y_true, y_pred)

        print("\n--- MÉTRICAS ---")
        print(f"N = {len(y_true)}")
        print(f"R²  = {r2:.4f}")
        print(f"RMSE = {rmse:.4f}")
        print(f"Correlação de Pearson (r) = {pearson_r:.4f}")
        print(f"P-valor = {pearson_p:.4e}")
        print("--------------------------------------\n")

        # GRÁFICO -----------------------
        plt.figure(figsize=(8, 8))

        plt.scatter(
            y_true,
            y_pred,
            s=50,
            alpha=0.7,
            edgecolors='black',
            linewidth=0.5
        )

        # Linha Y = X
        plt.plot([0, 20], [0, 20], 'r--', label='Y = X')

        # Linha de Tendência
        coef = np.polyfit(y_true, y_pred, 1)
        tendencia = np.poly1d(coef)
        x_vals = np.linspace(0, 20, 100)
        plt.plot(x_vals, tendencia(x_vals), 'b-', linewidth=2, label='Linha de Tendência')

        # Título
        plt.title(
            f'Real vs Predito\nR² = {r2:.4f} | Pearson r = {pearson_r:.4f}',
            fontsize=12
        )

        plt.xlabel(f'{COLUNA_REAL} (Real)')
        plt.ylabel(f'{COLUNA_PRED} (Predito)')

        plt.xlim([0, 20])
        plt.ylim([0, 20])

        plt.gca().set_aspect('equal', adjustable='box')
        plt.grid(True, linestyle=':')
        plt.legend()
        plt.tight_layout()
        plt.show()

except Exception as e:
    print(f"Erro: {e}")


Aqui métricas são apresentadas tanto individualmente para cada modelo dentro de cada grupo quanto de forma conjunta, combinando os grupos com e sem fator de risco, permitindo avaliar o desempenho global das equações e a concordância entre valores medidos e estimados. Esta célula foi desenvolvida com o fito de apresentar os resultados com mais clareza.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

ARQUIVO_ENTRADA = 'BANCOS/BAEPENDI_PADRAO.xlsx'
ARQUIVO_SAIDA = 'BAEPENDI_APLICACAO.xlsx'


def r2_ajustado(y_true, y_pred, p):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    if (n - p - 1) <= 0:
        return np.nan
    # return 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return r2


def icc_3_1(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(y_true)
    if n < 2:
        return np.nan
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    ss_subject = np.sum((y_true - mean_true) ** 2 + (y_pred - mean_pred) ** 2)
    ss_error = np.sum((y_true - y_pred) ** 2)
    MSR = ss_subject / (n - 1)
    MSE = ss_error / (2 * (n - 1))
    return (MSR - MSE) / (MSR + MSE)


def calcular_metricas_individual(df, coluna_predita, nome_grupo, p):
    df_clean = df.dropna(subset=['VOP', coluna_predita])
    y_true = df_clean['VOP']
    y_pred = df_clean[coluna_predita]
    if len(y_true) < 3:
        return None
    r2 = r2_score(y_true, y_pred)
    r2_adj = r2_ajustado(y_true, y_pred, p)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    icc = icc_3_1(y_true, y_pred)
    print(f"| {nome_grupo:<25} | {coluna_predita:<25} | {len(y_true):<5} | {r2_adj:.4f} | {rmse:.4f} | {icc:.4f} |")
    return r2, r2_adj, rmse, icc


def calcular_metricas_total(df0, df1, col_neg, col_pos, conjunto_nome, p_total):
    df_temp_0 = df0.dropna(subset=['VOP', col_neg])[['VOP', col_neg]].rename(columns={col_neg: 'Y_PRED'})
    df_temp_1 = df1.dropna(subset=['VOP', col_pos])[['VOP', col_pos]].rename(columns={col_pos: 'Y_PRED'})
    df_total = pd.concat([df_temp_0, df_temp_1], ignore_index=True)
    y_true = df_total['VOP']
    y_pred = df_total['Y_PRED']
    n = len(y_true)
    if n < 3:
        return None
    vop_medio = y_true.mean()
    r2 = r2_score(y_true, y_pred)
    r2_adj = r2_ajustado(y_true, y_pred, p_total)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    icc = icc_3_1(y_true, y_pred)
    print(f"| {conjunto_nome:<10} | {vop_medio:.4f} | {r2:.4f} | {r2_adj:.4f} | {rmse:.4f} | {icc:.4f} | {n} |")
    return vop_medio, r2, r2_adj, rmse, icc


df_completo = pd.read_excel(ARQUIVO_ENTRADA)

df0 = df_completo[df_completo['FATOR_RISCO'] == 0].copy()
df1 = df_completo[df_completo['FATOR_RISCO'] == 1].copy()

age0, MAP0, sexo0 = df0['IDADE'], df0['PAM'], df0['SEXO']
age1, MAP1, sexo1 = df1['IDADE'], df1['PAM'], df1['SEXO']

fator_sexo1_pos = np.where(sexo1 == 1, 0.0929, -0.0929)

df0['Eq_ufes_sem_sexo_(-)'] = (
        4.43 - 0.052 * age0 + 0.00079 * age0 ** 2
        + 0.00036 * age0 * MAP0 + 0.0235 * MAP0
)

df0['Eq_europa_(-)'] = (
        4.62 - 0.13 * age0 + 0.0018 * age0 ** 2
        + 0.0006 * age0 * MAP0 + 0.0284 * MAP0
)

df0['Eq_ufes_com_sexo_(-)'] = (
        5.56 - 0.065 * age0 + 0.00071 * age0 ** 2
        + 0.00058 * age0 * MAP0 + 0.0101 * MAP0
        + np.where(sexo0 == 1, 0.188, -0.188)
)

df1['Eq_ufes_sem_sexo_(+)'] = (
        6.52 - 0.173 * age1 + 0.002319 * age1 ** 2
        - 0.00001355 * age1 ** 2 * MAP1
        + 0.001430 * age1 * MAP1 + 0.005191 * MAP1
)

df1['Eq_ufes_com_sexo_(+)'] = (
        6.73 - 0.171 * age1 + 0.002267 * age1 ** 2
        - 0.00001331 * age1 ** 2 * MAP1
        + 0.001447 * age1 * MAP1 + 0.002360 * MAP1
        + fator_sexo1_pos
)

df1['Eq_europa_(+)'] = (
        9.58 - 0.402 * age1 + 0.004560 * age1 ** 2
        - 0.0000262 * age1 ** 2 * MAP1
        + 0.003176 * age1 * MAP1 - 0.01832 * MAP1
)

print("| Grupo                     | Modelo                         | N     | R²_adj | RMSE   | ICC    |")
print("|---------------------------|--------------------------------|-------|--------|--------|--------|")

calcular_metricas_individual(df0, 'Eq_ufes_sem_sexo_(-)', 'Sem Fator (UFES SEM -)', 4)
calcular_metricas_individual(df0, 'Eq_europa_(-)', 'Sem Fator (EUROPA -)', 4)
calcular_metricas_individual(df0, 'Eq_ufes_com_sexo_(-)', 'Sem Fator (UFES COM -)', 5)

calcular_metricas_individual(df1, 'Eq_ufes_sem_sexo_(+)', 'Com Fator (UFES SEM +)', 4)
calcular_metricas_individual(df1, 'Eq_ufes_com_sexo_(+)', 'Com Fator (UFES COM +)', 5)
calcular_metricas_individual(df1, 'Eq_europa_(+)', 'Com Fator (EUROPA +)', 4)

print("\n| Conjunto   | VOP Médio | R² simples | R² adj. total | RMSE total | ICC total | N |")
print("|------------|-----------|------------|---------------|------------|-----------|---|")

calcular_metricas_total(df0, df1, 'Eq_europa_(-)', 'Eq_europa_(+)', 'EUROPA', 4)
calcular_metricas_total(df0, df1, 'Eq_ufes_sem_sexo_(-)', 'Eq_ufes_sem_sexo_(+)', 'SEM_SEXO', 4)
calcular_metricas_total(df0, df1, 'Eq_ufes_com_sexo_(-)', 'Eq_ufes_com_sexo_(+)', 'COM_SEXO', 5)

# with pd.ExcelWriter(ARQUIVO_SAIDA, engine='xlsxwriter') as writer:
#     df_completo.to_excel(writer, sheet_name='Dados_Originais', index=False)
#     df0.to_excel(writer, sheet_name='Sem_Fator', index=False)
#     df1.to_excel(writer, sheet_name='Com_Fator', index=False)